In [10]:
# ── Cell 1：OrderBook 类定义 ──────────────────────────────────────────────────
_PRICE_SCALE = 1000

def _to_int_price(price):
    return round(price * _PRICE_SCALE)

def _to_float_price(int_price):
    return int_price / _PRICE_SCALE


class OrderBook:
    LEVELS = 10

    def __init__(self):
        self.bids = {}
        self.asks = {}
        self.last_price = 0.0
        self.cum_volume = 0
        self.cum_amount = 0.0
        self.timestamp  = 0
        self._bp1_int   = 0
        self._ap1_int   = 0
        self.bv1        = 0
        self.av1        = 0
        self.spread     = 0.0
        self.mid_price  = 0.0
        self.imbalance  = 0.0

    @property
    def bp1(self):
        return _to_float_price(self._bp1_int)

    @property
    def ap1(self):
        return _to_float_price(self._ap1_int)

    def _update_metrics(self):
        bid_empty = (self._bp1_int == 0)
        ask_empty = (self._ap1_int == 0)
        if bid_empty or ask_empty:
            self.spread    = float('nan')
            self.mid_price = float('nan')
        else:
            bp = self.bp1
            ap = self.ap1
            self.spread    = round(ap - bp, 6)
            self.mid_price = round((bp + ap) / 2, 6)
        total = self.bv1 + self.av1
        self.imbalance = round((self.bv1 - self.av1) / total, 6) if total > 0 else 0.0

    def init_from_snapshot(self, row):
        self.bids.clear()
        self.asks.clear()
        for i in range(1, self.LEVELS + 1):
            price = float(row[f'bp{i}'])
            vol   = int(row[f'bv{i}'])
            if vol > 0 and price > 0:
                self.bids[_to_int_price(price)] = vol
        for i in range(1, self.LEVELS + 1):
            price = float(row[f'ap{i}'])
            vol   = int(row[f'av{i}'])
            if vol > 0 and price > 0:
                self.asks[_to_int_price(price)] = vol
        self.last_price = float(row['last'])
        self.cum_volume = int(row['volume'])
        self.cum_amount = float(row['amount'])
        self.timestamp  = int(row.name)
        self._bp1_int   = _to_int_price(float(row['bp1']))
        self._ap1_int   = _to_int_price(float(row['ap1']))
        self.bv1        = int(row['bv1'])
        self.av1        = int(row['av1'])
        self._update_metrics()

    def apply_order(self, row):
        if str(getattr(row, 'order_type')) != 'limit':
            return
        side     = str(getattr(row, 'side'))
        int_p    = _to_int_price(float(getattr(row, 'price')))
        quantity = int(getattr(row, 'quantity'))
        if side == 'buy':
            self.bids[int_p] = self.bids.get(int_p, 0) + quantity
            if int_p > self._bp1_int:
                self._bp1_int = int_p
                self.bv1      = quantity
                self._update_metrics()
            elif int_p == self._bp1_int:
                self.bv1 += quantity
                self._update_metrics()
        else:
            self.asks[int_p] = self.asks.get(int_p, 0) + quantity
            if self._ap1_int == 0 or int_p < self._ap1_int:
                self._ap1_int = int_p
                self.av1      = quantity
                self._update_metrics()
            elif int_p == self._ap1_int:
                self.av1 += quantity
                self._update_metrics()
        self.timestamp = int(row.Index)

    def apply_trade(self, row):
        price    = float(getattr(row, 'price'))
        quantity = int(getattr(row, 'quantity'))
        int_p    = _to_int_price(price)
        if int_p <= self._bp1_int:
            book, hit_top, is_bid_side = self.bids, (int_p == self._bp1_int), True
        elif int_p >= self._ap1_int and self._ap1_int != 0:
            book, hit_top, is_bid_side = self.asks, (int_p == self._ap1_int), False
        else:
            bid_empty = (self._bp1_int == 0)
            ask_empty = (self._ap1_int == 0)
            if bid_empty and ask_empty:
                return
            elif bid_empty:
                book, hit_top, is_bid_side = self.asks, (int_p == self._ap1_int), False
            elif ask_empty:
                book, hit_top, is_bid_side = self.bids, (int_p == self._bp1_int), True
            else:
                dist_bid = int_p - self._bp1_int
                dist_ask = self._ap1_int - int_p
                if dist_bid <= dist_ask:
                    book, hit_top, is_bid_side = self.bids, (int_p == self._bp1_int), True
                else:
                    book, hit_top, is_bid_side = self.asks, (int_p == self._ap1_int), False
        top_cleared = False
        if int_p in book:
            book[int_p] = max(0, book[int_p] - quantity)
            if book[int_p] == 0:
                del book[int_p]
                top_cleared = hit_top
        self.last_price  = price
        self.cum_volume += quantity
        self.cum_amount += price * quantity
        if is_bid_side:
            if top_cleared:
                if self.bids:
                    self._bp1_int = max(self.bids)
                    self.bv1      = self.bids[self._bp1_int]
                else:
                    self._bp1_int, self.bv1 = 0, 0
            elif hit_top:
                self.bv1 = self.bids.get(self._bp1_int, 0)
        else:
            if top_cleared:
                if self.asks:
                    self._ap1_int = min(self.asks)
                    self.av1      = self.asks[self._ap1_int]
                else:
                    self._ap1_int, self.av1 = 0, 0
            elif hit_top:
                self.av1 = self.asks.get(self._ap1_int, 0)
        self._update_metrics()
        self.timestamp = int(row.Index)

    def to_snapshot(self):
        sorted_bids = sorted(self.bids.keys(), reverse=True)[:self.LEVELS]
        sorted_asks = sorted(self.asks.keys())[:self.LEVELS]
        result = {
            'timestamp':  self.timestamp,
            'last_price': self.last_price,
            'cum_volume': self.cum_volume,
            'cum_amount': round(self.cum_amount, 2),
            'spread':     self.spread,
            'mid_price':  self.mid_price,
            'imbalance':  self.imbalance,
        }
        for i in range(self.LEVELS):
            if i < len(sorted_bids):
                p = sorted_bids[i]
                result[f'bp{i+1}'] = _to_float_price(p)
                result[f'bv{i+1}'] = self.bids[p]
            else:
                result[f'bp{i+1}'] = 0.0
                result[f'bv{i+1}'] = 0
        for i in range(self.LEVELS):
            if i < len(sorted_asks):
                p = sorted_asks[i]
                result[f'ap{i+1}'] = _to_float_price(p)
                result[f'av{i+1}'] = self.asks[p]
            else:
                result[f'ap{i+1}'] = 0.0
                result[f'av{i+1}'] = 0
        return result

print('✓ OrderBook 类定义完成')

✓ OrderBook 类定义完成


In [11]:
# ── Cell 2：DataLoader 类定义 ─────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np

_EVENT_PRIORITY = {'order': 0, 'trade': 1}

class DataLoader:
    def __init__(self, clean_dir):
        self.clean_dir   = clean_dir
        self._df_level2  = None
        self._df_order   = None
        self._df_trade   = None

    def _lazy_load(self, attr, filename):
        if getattr(self, attr) is None:
            path = os.path.join(self.clean_dir, filename)
            df   = pd.read_parquet(path, engine='pyarrow')
            setattr(self, attr, df)
            print(f'[DataLoader] {filename} 已加载：{len(df):,} 行')
        return getattr(self, attr)

    @property
    def level2(self):
        return self._lazy_load('_df_level2', 'level2.parquet')

    @property
    def order(self):
        return self._lazy_load('_df_order', 'order.parquet')

    @property
    def trade(self):
        return self._lazy_load('_df_trade', 'trade.parquet')

    def find_anchor(self, t_target):
        df  = self.level2
        idx = df.index.searchsorted(t_target, side='right') - 1
        if idx < 0:
            raise ValueError(f'找不到锚点：t_target={t_target} 早于最早时间 {df.index[0]}')
        return df.iloc[idx]

    def robust_time_parse(self, series):  # 加上 self 变为类方法
        """
        确保 0 行过滤：兼容 HH:MM:SS.m, HH:MM:SS:mm, HH:MM:SS.mmm, HH:MM:SS.uuuuuu 等所有格式
        """
        # 1. 强制转为字符串并提取：时、分、秒、以及后面的数字部分
        parts = series.astype(str).str.extract(r'(\d{2}):(\d{2}):(\d{2})[.:](\d+)')
    
        # 2. 如果提取完全失败（防止整行 NaN），填充默认值
        parts = parts.fillna("0")
    
        # 3. 毫秒处理逻辑：
        # 取前3位。如果是 "1"，ljust(3, '0') 变成 "100"；如果是 "123456"，str[:3] 变成 "123"
        ms = parts[3].str[:3].str.ljust(3, '0')
    
        # 4. 拼合为 HHMMSSmmm 格式的整数
        combined_time = (parts[0] + parts[1] + parts[2] + ms).astype(np.int64)
        return combined_time
    
    def get_events(self, t_start, t_end):
        """
        提取 [t_start, t_end] 期间的流水，并进行深度稳健性处理
        """
        # --- 1. 处理 order 表 ---
        df_o = self.order.copy()
        # 修复：加上 self. 调用类内部方法
        df_o['time_int'] = self.robust_time_parse(df_o['order_time'])
    
        # 过滤时间区间
        df_order = df_o.loc[(df_o['time_int'] > t_start) & (df_o['time_int'] <= t_end)].copy()
        df_order['event_type'] = 'order'
        df_order['unified_time'] = df_order['time_int']

        # --- 2. 处理 trade 表 ---
        df_t = self.trade.copy()
        # 修复：加上 self. 调用类内部方法
        df_t['time_int'] = self.robust_time_parse(df_t['trade_time'])
    
        # 过滤时间区间
        df_trade = df_t.loc[(df_t['time_int'] > t_start) & (df_t['time_int'] <= t_end)].copy()
        df_trade['event_type'] = 'trade'
        df_trade['unified_time'] = df_trade['time_int']

        # --- 3. 合并与排序 ---
        combined = pd.concat([df_order, df_trade], axis=0, sort=False)
    
        if combined.empty:
            print(f"⚠️ 警告：在区间 [{t_start}, {t_end}] 内未找到任何流水")
            return combined

        # 优先级：先排 order，后排 trade
        combined['priority'] = combined['event_type'].map(_EVENT_PRIORITY)

        # 排序逻辑：时间 -> 序号(如果有) -> 优先级
        sort_cols = ['unified_time']
        if 'seq_no' in combined.columns:
            combined['seq_no'] = combined['seq_no'].fillna(0).astype(np.int64)
            sort_cols.append('seq_no')
        sort_cols.append('priority')

        combined.sort_values(sort_cols, kind='stable', inplace=True)
    
        # 设置索引
        combined.set_index('unified_time', inplace=True)
        combined.index.name = 'time'

        # 清理临时列
        cols_to_drop = ['time_int', 'priority']
        combined.drop(columns=[c for c in cols_to_drop if c in combined.columns], inplace=True)

        return combined

print('✓ DataLoader 类定义完成（已内置稳健解析逻辑）')

✓ DataLoader 类定义完成（已内置稳健解析逻辑）


In [12]:
# ── Cell 3：环境配置 & 初始化 ─────────────────────────────────────────────────
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)


CLEAN_DIR = os.path.abspath(os.path.join(os.getcwd(), 'clean_data'))

print(f'CLEAN_DIR: {CLEAN_DIR}')
loader = DataLoader(CLEAN_DIR)
ob     = OrderBook()

loader = DataLoader(CLEAN_DIR)
ob     = OrderBook()

print(f'CLEAN_DIR: {CLEAN_DIR}')
print('✓ loader 和 ob 初始化完成')

CLEAN_DIR: /Users/chongjidelaoshu/Desktop/my-first-project/clean_data
CLEAN_DIR: /Users/chongjidelaoshu/Desktop/my-first-project/clean_data
✓ loader 和 ob 初始化完成


In [13]:
# ── Cell 4：寻找时空锚点 ──────────────────────────────────────────────────────
T_TARGET = 93501000   # 目标时间戳，例如 09:35:00.000

anchor = loader.find_anchor(T_TARGET)
ob.init_from_snapshot(anchor)

print(f'目标时间   T_target : {T_TARGET}')
print(f'底座时间   T_start  : {ob.timestamp}')
print(f'距目标时间           : {T_TARGET - ob.timestamp} ms')

[DataLoader] level2.parquet 已加载：5,005 行
目标时间   T_target : 93501000
底座时间   T_start  : 93501000
距目标时间           : 0 ms


In [14]:
# ── Cell 5：提取与检查流水 ────────────────────────────────────────────────────
T_START = ob.timestamp

T_END = 93600000  

print(f"🎯 引擎当前时间 T_START : {T_START}")
print(f"🎯 引擎目标时间 T_END   : {T_END}")

# 注意这里要把原来的 T_TARGET 换成新的 T_END
events  = loader.get_events(T_START, T_END)

print(f'\n流水总笔数：{len(events):,}')
print('\n── head(10) ──')
print(events.head(10))

🎯 引擎当前时间 T_START : 93501000
🎯 引擎目标时间 T_END   : 93600000
[DataLoader] order.parquet 已加载：304,899 行
[DataLoader] trade.parquet 已加载：79,528 行

流水总笔数：3,160

── head(10) ──
                 id      date  channel    index  price                        order_time     symbol  quantity  side                    number market      other_info order_type  year  month  day  \
time                                                                                                                                                                                                
93501030  538282370  20240207        4   746663 2.3450  2024-02-07 09:35:01.030000+08:00  SH.510050    310000   buy   20240207_4_767905_place     sh    [['0', '0']]      place  2024      2    7   
93501030  538282369  20240207        4   746662 2.3460  2024-02-07 09:35:01.030000+08:00  SH.510050    310000   buy  20240207_4_765549_cancel     sh    [['0', '0']]     cancel  2024      2    7   
93501040  538282372  20240207        4   74671

In [15]:
# ── Cell 6：重放流水，重建盘口 ────────────────────────────────────────────────

import pandas as pd
import time

# ── 步骤 1：初始化"记录本" ────────────────────────────────────────────────────
spread_records         = []
trade_metrics          = []
mid_price_ts           = []

total_submitted_volume = 0.0
total_traded_volume    = 0.0

# ── 步骤 2-4：主循环 ──────────────────────────────────────────────────────────
t0 = time.time()

for row in events.itertuples():

    # ---------- 应用事件，更新盘口 ----------
    if row.event_type == 'order':
        ob.apply_order(row)

        # 仅新增挂单计入委托量 (防止撤单也被当成新增委托)
        if getattr(row, 'order_type', '') == 'place':
            total_submitted_volume += float(getattr(row, 'quantity', 0) or 0)

    elif row.event_type == 'trade':
        ob.apply_trade(row)

        # 成交量累加
        total_traded_volume += float(getattr(row, 'quantity', 0) or 0)

    # ---------- 步骤 2：盘口快照 & 表观指标 ----------
    bid1 = ob.bp1
    ask1 = ob.ap1

    # 只有当盘口两边都有报价时，才记录（过滤掉开盘前或单边熔断的极端情况）
    if bid1 is not None and ask1 is not None and bid1 > 0 and ask1 > 0:
        mid    = ob.mid_price
        spread = ob.spread

        # 直接从 itertuples() 的 Index 获取统一的时间戳 (整数格式)
        ts = row.Index

        spread_records.append({
            'timestamp': ts,
            'mid':       mid,
            'spread':    spread,
            'bid1':      bid1,
            'ask1':      ask1,
        })

        mid_price_ts.append({'timestamp': ts, 'mid': mid})

        # ---------- 步骤 3：交易成本指标（仅 Trade 事件） ----------
        if row.event_type == 'trade':
            trade_price = float(getattr(row, 'price', 0) or 0)
            qty         = float(getattr(row, 'quantity', 0) or 0)

            # 1. 先尝试从 side 字段读取（兼容常见代号）
            direction = 0
            side_val = str(getattr(row, 'side', '')).strip().lower() if pd.notna(getattr(row, 'side', None)) else ''
            
            if side_val in ['buy', 'b', '1']:
                direction = 1
            elif side_val in ['sell', 's', '-1', '2']:
                direction = -1
            else:
                # 2. 如果 side 字段为空，用“盘口比对法”反推主动方向
                if trade_price >= ask1:
                    direction = 1  # 向上吃掉卖单，主动买
                elif trade_price <= bid1:
                    direction = -1 # 向下砸穿买单，主动卖
                else:
                    # 成交在买卖中间价，看偏向哪边
                    direction = 1 if trade_price > mid else -1

            # 记录交易指标
            if direction != 0 and mid > 0:
                effective_spread = 2 * direction * (trade_price - mid)
                slippage         = effective_spread / 2.0

                trade_metrics.append({
                    'timestamp':        ts,
                    'price':            trade_price,
                    'quantity':         qty,
                    'direction':        direction,
                    'mid_at_trade':     mid,
                    'effective_spread': effective_spread,
                    'slippage':         slippage,
                })

elapsed_ms = (time.time() - t0) * 1000
print(f'重放完成：{len(events):,} 笔流水，耗时 {elapsed_ms:.2f} ms')
print(f'平均每笔：{elapsed_ms / max(len(events), 1):.4f} ms')

# ── 步骤 4：订单成交率 ────────────────────────────────────────────────────────
fill_ratio = (total_traded_volume / total_submitted_volume
              if total_submitted_volume > 0 else float('nan'))
print(f'\n📊 全局 Order Fill Ratio：{fill_ratio:.4%}')
print(f'   委托量：{total_submitted_volume:,.0f}  |  成交量：{total_traded_volume:,.0f}')

# ── 步骤 5：事后计算市场冲击（Order Impact） ──────────────────────────────────
IMPACT_DELTA_MS = 100  # 观察交易发生后 100 毫秒的价格变化

df_mid    = pd.DataFrame(mid_price_ts)
df_trades = pd.DataFrame(trade_metrics)

if not df_mid.empty and not df_trades.empty:
    
    # 将 HHMMSSmmm 格式的整数解析为 Pandas 时间格式的函数
    def parse_time(series):
        # 补齐 9 位 (HHMMSSmmm) + 3 个 0 凑够微秒 (%f) 格式
        time_str = series.astype(str).str.zfill(9) + '000'
        return pd.to_datetime(time_str, format='%H%M%S%f')

    df_mid['timestamp']    = parse_time(df_mid['timestamp'])
    df_trades['timestamp'] = parse_time(df_trades['timestamp'])

    # 排序（merge_asof 的硬性要求）
    df_mid    = df_mid.sort_values('timestamp').reset_index(drop=True)
    df_trades = df_trades.sort_values('timestamp').reset_index(drop=True)

    # 加上时间偏移量
    df_trades['ts_after'] = df_trades['timestamp'] + pd.Timedelta(milliseconds=IMPACT_DELTA_MS)

    # 寻找交易发生 100ms 后的盘口中间价
    df_impact = pd.merge_asof(
        df_trades.sort_values('ts_after'),
        df_mid[['timestamp', 'mid']].rename(columns={'mid': 'mid_after'}),
        left_on='ts_after',
        right_on='timestamp',
        direction='forward',
    )

    # 计算订单冲击
    df_impact['order_impact'] = (df_impact['direction']
                                 * (df_impact['mid_after'] - df_impact['mid_at_trade']))

    print(f'\n📊 市场冲击统计（Δt = {IMPACT_DELTA_MS} ms）：')
    print(df_impact['order_impact'].describe().to_string())
else:
    print('\n⚠️ 未收集到足够的 Trade 数据来计算市场冲击。')

# ── 汇总 ──────────────────────────────────────────────────────────────────────
df_spread       = pd.DataFrame(spread_records)
df_trades_final = df_impact if 'df_impact' in dir() else df_trades

print(f'\n✅ spread_records 行数：{len(df_spread):,}')
print(f'✅ trade_metrics  行数：{len(df_trades_final):,}')

重放完成：3,160 笔流水，耗时 13.24 ms
平均每笔：0.0042 ms

📊 全局 Order Fill Ratio：6.0650%
   委托量：246,336,607  |  成交量：14,940,300

📊 市场冲击统计（Δt = 100 ms）：
count   743.0000
mean      0.0000
std       0.0000
min      -0.0000
25%      -0.0000
50%       0.0000
75%       0.0000
max       0.0005

✅ spread_records 行数：3,160
✅ trade_metrics  行数：743


In [16]:
def cross_validate_display_v2(ob_reconstructed, official_df, price_scale=1000.0):
    print("=" * 75)
    print(f"  重建盘口 vs 官方快照 (校验时刻: {ob_reconstructed.timestamp})")
    print("=" * 75)
    print(f"{'级别':<6} {'价格(重建)':>12} {'价格(官方)':>12} {'一致':<4} | {'数量(重建)':>12} {'数量(官方)':>12} {'一致':<4}")
    print("-" * 75)

    def get_top_n(data_dict, reverse=True):
        valid_prices = sorted([p for p, q in data_dict.items() if q > 0], reverse=reverse)
        return [(p / price_scale, data_dict[p]) for p in valid_prices[:10]]

    rec_bids = get_top_n(ob_reconstructed.bids, reverse=True)
    rec_asks = get_top_n(ob_reconstructed.asks, reverse=False)

    for i in range(1, 11):
        # --- 买单对比 ---
        off_bp = float(official_df.get(f'bp{i}', 0.0))
        off_bv = int(official_df.get(f'bv{i}', 0))
        rb_p, rb_v = rec_bids[i-1] if i <= len(rec_bids) else (0.0, 0)
        
        bp_match = "✓" if np.isclose(rb_p, off_bp, atol=1e-6) else "✗"
        bv_match = "✓" if int(rb_v) == off_bv else "✗"

        print(f"L{i:<2} {rb_p:>12.4f} {off_bp:>12.4f} {bp_match:<4} | {int(rb_v):>12,d} {off_bv:>12,d} {bv_match:<4}")

    print("-" * 75)
    # 计算缩放后的中间价和价差
    scaled_mid = ob_reconstructed.mid_price / price_scale
    scaled_spread = ob_reconstructed.spread / price_scale
    print(f"Spread: {scaled_spread:.4f} | Mid: {scaled_mid:.4f}")
    print("=" * 75)

# ── 1. 重置引擎 ──
T_START, T_EVAL = 93501000, 93504000  
ob.init_from_snapshot(loader.find_anchor(T_START))

# ── 2. 推进引擎 (请确保此时 DataLoader 里的解析逻辑已修复，警告行数为 0) ──
events_cv = loader.get_events(T_START, T_EVAL)
print(f"📊 提取流水: {len(events_cv)} 笔")

for row in events_cv.itertuples():
    if row.event_type == 'order': ob.apply_order(row)
    elif row.event_type == 'trade': ob.apply_trade(row)

# ── 3. 对账 ──
cross_validate_display_v2(ob, loader.find_anchor(T_EVAL), price_scale=1000.0)

📊 提取流水: 150 笔
  重建盘口 vs 官方快照 (校验时刻: 93503970)
级别           价格(重建)       价格(官方) 一致   |       数量(重建)       数量(官方) 一致  
---------------------------------------------------------------------------
L1        2.3470       2.3480 ✗    |    3,837,100    1,193,200 ✗   
L2        2.3460       2.3470 ✗    |    1,876,300    3,565,700 ✗   
L3        2.3450       2.3460 ✗    |    2,620,400    2,289,300 ✗   
L4        2.3440       2.3450 ✗    |    1,145,200    2,622,500 ✗   
L5        2.3430       2.3440 ✗    |      736,300      955,200 ✗   
L6        2.3420       2.3430 ✗    |    1,321,900      736,300 ✗   
L7        2.3410       2.3420 ✗    |      703,800    1,535,500 ✗   
L8        2.3400       2.3410 ✗    |    2,803,100      703,800 ✗   
L9        2.3390       2.3400 ✗    |      406,200    2,803,100 ✗   
L10       0.0000       2.3390 ✗    |            0      406,200 ✗   
---------------------------------------------------------------------------
Spread: 0.0000 | Mid: 0.0023


In [17]:
# ── Cell 7：高频微观结构可视化 ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. 设置画图风格与字体（防止中文显示为方块）
plt.style.use('seaborn-v0_8-darkgrid')
# Windows 用户请使用 'SimHei'，Mac 用户请使用 'Arial Unicode MS'
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] 
plt.rcParams['axes.unicode_minus'] = False

# 2. 准备数据并确保时间格式适合画图
df_s = df_spread.copy()
df_t = df_trades_final.copy()

# 防御性转换：确保 X 轴是标准的 datetime 格式，这样图表底部的刻度才会好看
def ensure_datetime(df, col='timestamp'):
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        time_str = df[col].astype(str).str.zfill(9) + '000'
        df[col] = pd.to_datetime(time_str, format='%H%M%S%f')
    return df

df_s = ensure_datetime(df_s)
df_t = ensure_datetime(df_t)

# 3. 创建画布：上下两张子图，共享时间 X 轴
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(16, 10), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

# ══════════════════════════════════════════════════════════════════════════════
# ── 图 1：盘口价格通道与大单狙击点 ──
# ══════════════════════════════════════════════════════════════════════════════

# 画出中间价折线
ax1.plot(df_s['timestamp'], df_s['mid'], color='dimgray', linewidth=1.5, label='Mid Price (中间价)', alpha=0.8)

# 用半透明的蓝色面积图填满买一和卖一之间的区域，形成“价差通道”
ax1.fill_between(df_s['timestamp'], df_s['bid1'], df_s['ask1'], color='royalblue', alpha=0.15, label='Bid-Ask Spread (价差通道)')

# 把真实的成交流水（Trade）当作气泡画上去
# 气泡大小 (s) 与成交量成正比，红色代表主动买入，绿色代表主动卖出
buys = df_t[df_t['direction'] == 1]
ax1.scatter(buys['timestamp'], buys['price'], 
            s=np.clip(buys['quantity'] / 500, 10, 500), # 限制气泡大小范围，防止遮挡
            color='crimson', alpha=0.7, edgecolors='white', label='主动买入 (Buy Initiated)')

sells = df_t[df_t['direction'] == -1]
ax1.scatter(sells['timestamp'], sells['price'], 
            s=np.clip(sells['quantity'] / 500, 10, 500), 
            color='forestgreen', alpha=0.7, edgecolors='white', label='主动卖出 (Sell Initiated)')

ax1.set_title('微观盘口演变与主动交易成交点', fontsize=16, fontweight='bold')
ax1.set_ylabel('Price (价格)', fontsize=12)
ax1.legend(loc='upper left', frameon=True, facecolor='white')

# ══════════════════════════════════════════════════════════════════════════════
# ── 图 2：表观价差 vs 有效价差 ──
# ══════════════════════════════════════════════════════════════════════════════

# 画出盘口显示的表观价差（Quoted Spread）
ax2.plot(df_s['timestamp'], df_s['spread'], color='purple', linewidth=1.2, label='表观价差 (Quoted Spread)')

# 叠加真实的有效价差散点（Effective Spread）
ax2.scatter(df_t['timestamp'], df_t['effective_spread'], color='darkorange', s=15, alpha=0.8, label='单笔有效价差 (Effective Spread)')

ax2.set_title('流动性成本动态：做市商报价 vs 真实吃单成本', fontsize=14)
ax2.set_ylabel('Spread (价差)', fontsize=12)
ax2.set_xlabel('Time (时间)', fontsize=12)
ax2.legend(loc='upper left', frameon=True, facecolor='white')

# 调整布局并展示
plt.tight_layout()
plt.show()

KeyError: 'timestamp'